# Trabajo Práctico Integrador - Introducción al Análisis de Datos

**Caso:** análisis y predicción de cancelaciones en reservas hoteleras.

> La empresa busca comprender qué factores influyen en la cancelación de reservas en hoteles urbanos y resort, analizando variables como fechas de estadía, tipo de cliente, canal de reserva, historial previo y tarifas, con el fin de identificar patrones y mejorar la gestión operativa.


## Desarrollado por

- **Apellido y nombre:** Matias Carro y Hugo Catalan
- **Comisión:** 11
- **Dataset asignado:** Dataset K
- **Entrega:** Primer Entrega - Semana 3
- **Fecha:** 

## Importación de librerías

### Librerias a utilizar:

- **pandas:** para cargar el dataset y trabajar con datos en forma de tablas (DataFrames).
- **numpy:** para realizar operaciones numéricas y manejar arreglos de forma eficiente.



In [1]:
import pandas as pd
import numpy as np

print("Versión de Pandas:", pd.__version__)
print("Versión de numpy:", np.__version__)
print("\nLibrerías cargadas correctamente.")


Versión de Pandas: 3.0.5
Versión de numpy: 2.5.2

Librerías cargadas correctamente.


## 1. Presentacion del problema

El caso de estudio se centra en el análisis de reservas hoteleras con el objetivo de comprender qué factores están asociados a la cancelación de estadías. El dataset asignado contiene información detallada de cada reserva, incluyendo tipo de hotel, fechas de llegada, duración de la estadía, composición del grupo, país de origen, canal de reserva, tipo de cliente, historial previo, tarifa promedio por noche y características operativas como depósito, agente, pedidos especiales y cambios realizados.


### Relación entre datos, información y conocimiento
En este trabajo partimos de los **datos** que son valores crudos del sistema de reservas: fechas, cantidades, categorías y códigos.  
Mediante el análisis exploratorio estos datos se convierten en **información**, como distribuciones, patrones y diferencias entre reservas canceladas y no canceladas.  
A partir de esa información generamos el **conocimiento** que nos permite entender el comportamiento de los clientes y detectar factores que podrían influir en la cancelación de una reserva.  

Esta relación es clave para el caso: los datos del hotel por sí solos no dicen nada, pero al transformarlos en información y luego interpretarlos, podemos identificar variables relevantes (como `lead_time`, `deposit_type` o `customer_type`) que ayudan a explicar por qué algunas reservas se cancelan y otras no.

### Ciclo de vida del análisis
Este trabajo se enmarca en el ciclo de vida del análisis de datos, que incluye:
1. Obtención del dataset asignado.  
2. Comprensión inicial del problema (cancelaciones hoteleras).  
3. Exploración y limpieza mínima (EDA).  
4. Transformación y preparación de variables relevantes.  
5. Interpretación y comunicación de resultados.

### Variable objetivo
La **variable objetivo** del análisis es **`is_canceled`**, que indica si la reserva fue cancelada (`1`) o no (`0`).  
Su distribución será calculada y analizada en las próximas secciones para comprender el comportamiento general del conjunto de datos y orientar las preguntas del análisis.

### Preguntas iniciales que orientan el trabajo
- ¿Qué características diferencian a las reservas canceladas de las no canceladas?  
- ¿Influyen el tipo de hotel o el canal de reserva en la cancelación?  
- ¿Las reservas con mayor anticipación (`lead_time`) presentan mayor probabilidad de cancelación?  
- ¿Los clientes con pedidos especiales o estacionamiento tienden a cancelar menos?  
- ¿Las políticas de depósito (`deposit_type`) reducen la cancelación?  
- ¿Existen segmentos de mercado con mayor riesgo de cancelación?  

## 2. Carga del dataset

Se carga el Dataset perteneciente a la comisión 11:


In [2]:
df = pd.read_csv("hotel booking TPI grupo K.csv")

print("Dataset cargado correctamente.")
print("\nVista de los primeros datos: ")
df.head()


Dataset cargado correctamente.

Vista de los primeros datos: 


,booking_id,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,arrival_date,stays_in_weekend_nights,...,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,HB-014269,Resort Hotel,0,17,2025,February,8,21,2025-02-21,0,...,E,0,No Deposit,NaN,292.0,0,Transient,58.00,1,1
1,HB-018087,Resort Hotel,0,6,2023,November,44,1,2023-11-01,2,...,D,3,No Deposit,240.0,NaN,0,Transient,58.00,0,2
2,HB-022870,Resort Hotel,0,45,2024,April,15,8,2024-04-08,0,...,D,1,No Deposit,240.0,NaN,0,Transient-Party,65.00,0,2
3,HB-048154,City Hotel,0,95,2024,March,11,17,2024-03-17,2,...,A,0,No Deposit,9.0,NaN,0,Transient,73.95,0,1
4,HB-060351,City Hotel,1,277,2024,November,45,7,2024-11-07,1,...,A,0,Non Refund,NaN,NaN,0,Transient,100.00,0,0


## 4. Estructura general:

Filas y Columnas:

In [3]:
filas, columnas = df.shape
print(f"El dataset tiene {filas} filas y {columnas} columnas.")

print("Columnas del dataset:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.info())


El dataset tiene 25000 filas y 32 columnas.
Columnas del dataset:
['booking_id', 'hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'arrival_date', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']

Tipos de datos:
<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   booking_id                      25000 non-null  str    
 1   hotel                          

El dataset tiene 25.000 filas y 32 columnas.  
Las variables incluyen información temporal, categórica y numérica relevante para el análisis.


### Diccionario de variables 

| Variable | Traducción | Representación | Tipo de Datos |
|----------|------------|----------------|-----------|
| booking_id | ID de reserva | Identificador único de cada reserva. | str |
| hotel | Tipo de hotel | Indica si la reserva corresponde a City Hotel (Ciudad) o Resort Hotel (Resort). | str |
| is_canceled | Cancelada | Indica si la reserva fue cancelada (1) o no (0). | int64 |
| lead_time | Anticipación | Días entre la fecha de reserva y la fecha de llegada. | int64 |
| arrival_date_year | Año de llegada | Año en el que el huésped llega al hotel. | int64 |
| arrival_date_month | Mes de llegada | Mes en el que el huésped llega al hotel. | str |
| arrival_date_week_number | Semana de llegada | Número de semana del año en la que llega el huésped. | int64 |
| arrival_date_day_of_month | Día del mes de llegada | Día del mes en el que llega el huésped. | int64 |
| arrival_date | Fecha de llegada | Fecha completa de llegada (YYYY-MM-DD). | str |
| stays_in_weekend_nights | Noches de fin de semana | Cantidad de noches en fines de semana. | int64 |
| stays_in_week_nights | Noches de semana | Cantidad de noches de lunes a jueves. | int64 |
| adults | Adultos | Número de adultos en la reserva. | int64 |
| children | Niños | Número de niños en la reserva. | float64 |
| babies | Bebés | Número de bebés en la reserva. | int64 |
| meal | Tipo de comida | Plan de comidas asociado a la reserva (BB, HB, SC, etc.). | str |
| country | País | País de origen del huésped. | str |
| market_segment | Segmento de mercado | Tipo de cliente según el canal de adquisición. | str |
| distribution_channel | Canal de distribución | Canal por el cual se realizó la reserva. | str |
| is_repeated_guest | Huésped repetido | Indica si el cliente ya se alojó anteriormente. | int64 |
| previous_cancellations | Cancelaciones previas | Cantidad de reservas previas canceladas por el cliente. | int64 |
| previous_bookings_not_canceled | Reservas previas no canceladas | Cantidad de reservas previas completadas por el cliente. | int64 |
| reserved_room_type | Habitación reservada | Tipo de habitación solicitada originalmente. | str |
| assigned_room_type | Habitación asignada | Tipo de habitación finalmente asignada. | str |
| booking_changes | Cambios en la reserva | Número de modificaciones realizadas a la reserva. | int64 |
| deposit_type | Tipo de depósito | Política de depósito aplicada. | str |
| agent | Agente | Código del agente que gestionó la reserva. | float64 |
| company | Compañía | Código de la empresa asociada a la reserva. | float64 |
| days_in_waiting_list | Días en lista de espera | Tiempo que la reserva permaneció en espera antes de confirmarse. | int64 |
| customer_type | Tipo de cliente | Clasificación del cliente (Transient, Contract, Group, etc.). | str |
| adr | Tarifa promedio diaria | Precio promedio por noche de la reserva. | float64 |
| required_car_parking_spaces | Estacionamiento requerido | Cantidad de espacios de estacionamiento solicitados. | int64 |
| total_of_special_requests | Pedidos especiales | Número de solicitudes especiales realizadas por el cliente. | int64 |



## Tipos de variables 

Clasificacion de las variables por su tipo de datos

In [4]:
numericas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cadenas = df.select_dtypes(include=['str']).columns.tolist()

print("Cantidad de variables numéricas:", len(numericas))
print("Cantidad de variables de texto / fecha:", len(cadenas))



Cantidad de variables numéricas: 20
Cantidad de variables de texto / fecha: 12


### Clasificación inicial de las variables

| Variable | Tipo | Representación |
|----------|------|----------------|
| booking_id | Categórica (ID) | Identificador único de reserva. |
| hotel | Categórica | Tipo de hotel. |
| is_canceled | Numérica (booleana) | Indica si la reserva fue cancelada. |
| lead_time | Numérica | Días de anticipación de la reserva. |
| arrival_date_year | Numérica | Año de llegada. |
| arrival_date_month | Categórica | Mes de llegada. |
| arrival_date_week_number | Numérica | Semana del año. |
| arrival_date_day_of_month | Numérica | Día del mes. |
| arrival_date | Temporal | Fecha completa de llegada. |
| stays_in_weekend_nights | Numérica | Noches de fin de semana. |
| stays_in_week_nights | Numérica | Noches de semana. |
| adults | Numérica | Cantidad de adultos. |
| children | Numérica | Cantidad de niños. |
| babies | Numérica | Cantidad de bebés. |
| meal | Categórica | Tipo de comida. |
| country | Categórica | País de origen. |
| market_segment | Categórica | Segmento de mercado. |
| distribution_channel | Categórica | Canal de distribución. |
| is_repeated_guest | Numérica (booleana) | Indica si el huésped ya se alojó antes. |
| previous_cancellations | Numérica | Cancelaciones previas. |
| previous_bookings_not_canceled | Numérica | Reservas previas no canceladas. |
| reserved_room_type | Categórica | Habitación reservada. |
| assigned_room_type | Categórica | Habitación asignada. |
| booking_changes | Numérica | Cambios realizados a la reserva. |
| deposit_type | Categórica | Política de depósito. |
| agent | Categórica (ID) | Código del agente. |
| company | Categórica (ID) | Código de la compañía. |
| days_in_waiting_list | Numérica | Días en lista de espera. |
| customer_type | Categórica | Tipo de cliente. |
| adr | Numérica | Tarifa promedio diaria. |
| required_car_parking_spaces | Numérica | Espacios de estacionamiento solicitados. |
| total_of_special_requests | Numérica | Cantidad de pedidos especiales. |

---


### Resumen descriptivo

In [5]:
df.describe().round(2)

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
count,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,21511.00,1426.00,25000.00,25000.00,25000.00,25000.00
mean,0.37,103.29,2024.16,26.51,15.83,0.92,2.48,1.85,0.10,0.01,0.03,0.08,0.13,0.22,86.05,189.79,2.40,101.86,0.06,0.57
std,0.48,106.59,0.71,13.40,8.81,0.99,1.88,0.58,0.39,0.12,0.18,0.79,1.43,0.63,110.38,132.42,17.56,48.03,0.24,0.80
min,0.00,0.00,2023.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,9.00,0.00,-6.38,0.00,0.00
25%,0.00,18.00,2024.00,16.00,8.00,0.00,1.00,2.00,0.00,0.00,0.00,0.00,0.00,0.00,9.00,62.00,0.00,69.50,0.00,0.00
50%,0.00,68.00,2024.00,27.00,16.00,1.00,2.00,2.00,0.00,0.00,0.00,0.00,0.00,0.00,14.00,174.00,0.00,95.00,0.00,0.00
75%,1.00,159.00,2025.00,37.00,24.00,2.00,3.00,2.00,0.00,0.00,0.00,0.00,0.00,0.00,229.00,277.00,0.00,126.00,0.00,1.00
max,1.00,629.00,2025.00,52.00,31.00,14.00,34.00,50.00,3.00,10.00,1.00,26.00,66.00,14.00,531.00,539.00,391.00,510.00,3.00,5.00


In [6]:
df['booking_id'].value_counts()

booking_id
HB-014269    1
HB-018087    1
HB-022870    1
HB-048154    1
HB-060351    1
            ..
HB-088693    1
HB-102568    1
HB-061499    1
HB-075900    1
HB-078303    1
Name: count, Length: 25000, dtype: int64

## Caracterización descriptiva inicial:


### 1. booking_id


In [7]:


print(df["booking_id"].describe())
ids_unicos = df["booking_id"].nunique()
faltantes = df["booking_id"].isnull().sum()

print("\n Descripción:")

print(f"\nCantidad de faltantes: {faltantes}")
print (f"\nSe trata de {ids_unicos} IDs unicas")

count         25000
unique        25000
top       HB-014269
freq              1
Name: booking_id, dtype: object

 Descripción:

Cantidad de faltantes: 0

Se trata de 25000 IDs unicas


**Descripción:**  
Es un identificador único, no aporta información estadística. Se usa solo para referencia.

**Cantidad y Faltantes:** 
Se cuenta con 25000 IDs unicas, sin ninguna faltante

---

### 2. hotel

In [8]:

print(df["hotel"].shape)

conteo_total = df["hotel"].count()

faltantes = df["hotel"].isnull().sum()
porcentaje_faltantes = (faltantes / len(df)) * 100

distintos = df["hotel"].nunique()

conteo = df["hotel"].value_counts()
porcentaje = df["hotel"].value_counts(normalize=True) * 100

print("Datos totales:", conteo_total)
print("Faltantes:", faltantes)
print("Porcentaje de faltantes:", porcentaje_faltantes, "%")

print("Cantidad de valores distintos:", distintos)

print("\nDistribución:")
print(conteo)

print("\nPorcentaje:")
print(porcentaje)



(25000,)
Datos totales: 25000
Faltantes: 0
Porcentaje de faltantes: 0.0 %
Cantidad de valores distintos: 2

Distribución:
hotel
City Hotel      16716
Resort Hotel     8284
Name: count, dtype: int64

Porcentaje:
hotel
City Hotel      66.864
Resort Hotel    33.136
Name: proportion, dtype: float64


**Descripción:**  
La variable `hotel` indica el tipo de establecimiento donde se realizó la reserva. Es útil para comparar comportamientos entre City Hotel y Resort Hotel, ya que cada uno puede tener patrones diferentes de demanda, estacionalidad y cancelaciones.

**Faltantes:**  
0 (0%)

**Valores distintos:**  
2

**Distribución:**  
- City Hotel: 67%  
- Resort Hotel: 33%

**Observación:**  
La mayoría de las reservas corresponden al City Hotel. 

---

### 3. is_canceled

In [9]:
conteo_total = df["is_canceled"].count()

faltantes = df["is_canceled"].isnull().sum()
porcentaje_faltantes = (faltantes / len(df)) * 100

distintos = df["is_canceled"].nunique()

conteo = df["is_canceled"].value_counts()
porcentaje = df["is_canceled"].value_counts(normalize=True) * 100

print("Datos totales:", conteo_total)
print("Faltantes:", faltantes)
print("Porcentaje de faltantes:", porcentaje_faltantes, "%")

print("Cantidad de valores distintos:", distintos)

print("\nDistribución:")
print(conteo)

print("\nPorcentaje:")
print(porcentaje)




Datos totales: 25000
Faltantes: 0
Porcentaje de faltantes: 0.0 %
Cantidad de valores distintos: 2

Distribución:
is_canceled
0    15729
1     9271
Name: count, dtype: int64

Porcentaje:
is_canceled
0    62.916
1    37.084
Name: proportion, dtype: float64


### 3. is_canceled

**Descripción:**  
La variable `is_canceled` indica si la reserva fue cancelada (`1`) o no (`0`). Es una variable clave porque representa el resultado final del proceso de reserva y suele ser la variable objetivo en modelos de predicción de cancelaciones.

**Faltantes:**  
0 (0%)

**Valores distintos:**  
2  
- `0`: reserva no cancelada  
- `1`: reserva cancelada  

**Distribución:**  
- No canceladas: ~60%  
- Canceladas: ~40%

**Observación:**  
El dataset presenta una proporción considerable de cancelaciones. Esta distribución es importante para evaluar y llegar a una conclusion del motivo tan alto de las cancelaciones.

---

### 4. lead_time

In [10]:
conteo_total = df["lead_time"].count()
faltantes = df["lead_time"].isnull().sum()
porcentaje_faltantes = (faltantes / len(df)) * 100
distintos = df["lead_time"].nunique()

print("\nDatos totales:", conteo_total)
print(f"Faltantes: {faltantes}")
print(f"Porcentaje de faltantes: {porcentaje_faltantes}%")
print(f"Cantidad de valores distintos: {distintos}")


print("\nEstadisticas:")
print(df["lead_time"].describe())







Datos totales: 25000
Faltantes: 0
Porcentaje de faltantes: 0.0%
Cantidad de valores distintos: 460

Estadisticas:
count    25000.000000
mean       103.285280
std        106.585179
min          0.000000
25%         18.000000
50%         68.000000
75%        159.000000
max        629.000000
Name: lead_time, dtype: float64



**Descripción:**  
La variable lead_time representa la cantidad de días entre la fecha de reserva y la fecha de llegada (anticipacion). 

**Faltantes:**  
0 

Se trata de 460 valores diferentes. De un total de 25000 datos, sin ningun faltante.

**Hay una endencia central:** La media es 103 días, pero la mediana es 68, esto nos dice que la distribución está sesgada por valores extremos altos.

**Dispersión:** La desviación estándar es 106 días, bastante elevada, lo que confirma gran variabilidad.

**Rango:** Va de 0 (reservas hechas el mismo día) hasta 629 días (más de un año y medio de anticipación).

Con los **cuartiles** sabemos que:

  - 25% de las reservas se hacen con menos de 18 días de anticipación.

  - 50% con menos de 68 días.

  - 75% con menos de 159 días.

---


### 5. arrival_date_year


In [11]:
conteo_total = df["arrival_date_year"].count()

print("\nEstadisticas:")
print(df["arrival_date_year"].describe())

print("\nDatos y Cantidades:")
print(df["arrival_date_year"].value_counts())

print("\nPorcentajes:")
print(df["arrival_date_year"].value_counts(normalize=True))


Estadisticas:
count    25000.000000
mean      2024.158880
std          0.706072
min       2023.000000
25%       2024.000000
50%       2024.000000
75%       2025.000000
max       2025.000000
Name: arrival_date_year, dtype: float64

Datos y Cantidades:
arrival_date_year
2024    11906
2025     8533
2023     4561
Name: count, dtype: int64

Porcentajes:
arrival_date_year
2024    0.47624
2025    0.34132
2023    0.18244
Name: proportion, dtype: float64


**Descripcion:**
La variable `arrival_date_year` representa el año de llegada del huésped al hotel. 

**Faltantes:**
Está completa (0 faltantes) 

Contiene solo tres valores distintos: **2023**, **2024** y **2025**.

- El año 2024 concentra casi la mitad de las reservas (47,6 %), siendo el período con mas huespedes  del dataset.
- El año 2025 aporta un 34,1 %, mostrando continuidad hacia el futuro.
- El año 2023 tiene el 18,2 %, con menor peso porque es el inicio del registro.

Se deberia analizar el motivo por el cual en el 2024 hubo mayor cantidad de huespedes y esta bajo durante el 2025.

---

### 6. arrival_date_month 

In [ ]:
print(f"Estadisticas basicas: ")
print(df["arrival_date_month"].describe())

print("\nCantidad por mes:")
print(df["arrival_date_month"].value_counts())

print("\nPorcentajes por mes:")
print(df["arrival_date_month"].value_counts(normalize=True).round(2))

print(f"\n Meses de los que tenemos datos: ")
print(df["arrival_date_month"].unique())


Estadisticas basicas: 
count      25000
unique        12
top       August
freq        2861
Name: arrival_date_month, dtype: object

Cantidad por mes:
arrival_date_month
August       2861
July         2691
May          2448
April        2379
October      2356
June         2321
September    2165
March        2020
February     1723
December     1389
November     1370
January      1277
Name: count, dtype: int64

Porcentajes:
arrival_date_month
August       0.11
July         0.11
May          0.10
April        0.10
October      0.09
June         0.09
September    0.09
March        0.08
February     0.07
December     0.06
November     0.05
January      0.05
Name: proportion, dtype: float64

 Meses de los que tenemos datos: 
<StringArray>
[ 'February',  'November',     'April',     'March',    'August',       'May',
      'July', 'September',  'December',   'October',   'January',      'June']
Length: 12, dtype: str


**Descripción:**

La variable `arrival_date_month` representa el mes de llegada del huésped al hotel.
Faltantes:

**Faltantes:**
Está completa (0 faltantes).

Contiene 12 valores distintos, correspondientes a todos los meses del año.

- Los meses con mayor cantidad de huéspedes son Agosto (11%), Julio (11%) y Mayo (10%), mostrando la temporada alta.

- Meses como Enero (5%), Noviembre (5%) y Diciembre (6%) tienen un menor volumen de reservas.

Se debería analizar si esta distribucion de las reservas se relaciona directamente con estacionalidad o hay otros factores claves que influyan.

---

### 7. arrival_date_week_number

In [25]:

conteo_total = df["arrival_date_week_number"].count()
print("Conteo total:", conteo_total)

print("\nCantidad de valores distintos:")
print(f"{df["arrival_date_week_number"].nunique()} Semanas en el año")

print("\n Informacion:")
print(df["arrival_date_week_number"].info())

print("\nEstadísticas:")
print(df["arrival_date_week_number"].describe())

print("\nDatos y Cantidades:")
print(df["arrival_date_week_number"].value_counts().sort_index())

print("\nPorcentajes:")
print(df["arrival_date_week_number"].value_counts(normalize=True).round(4))




Conteo total: 25000

Cantidad de valores distintos:
52 Semanas en el año

 Informacion:
<class 'pandas.Series'>
RangeIndex: 25000 entries, 0 to 24999
Series name: arrival_date_week_number
Non-Null Count  Dtype
--------------  -----
25000 non-null  int64
dtypes: int64(1)
memory usage: 195.4 KB
None

Estadísticas:
count    25000.00000
mean        26.50996
std         13.39698
min          1.00000
25%         16.00000
50%         27.00000
75%         37.00000
max         52.00000
Name: arrival_date_week_number, dtype: float64

Datos y Cantidades:
arrival_date_week_number
1     343
2     200
3     323
4     338
5     300
6     323
7     462
8     436
9     549
10    385
11    454
12    511
13    439
14    485
15    538
16    552
17    594
18    598
19    516
20    591
21    609
22    513
23    581
24    550
25    522
26    550
27    578
28    614
29    634
30    612
31    594
32    703
33    735
34    589
35    578
36    471
37    527
38    562
39    471
40    571
41    565
42    533
43   

**Descripción:**

La variable `arrival_date_week_number` representa la semana del año en la que el huésped llega al hotel, con valores que van del 1 al 52.
Faltantes:

**Faltantes:**
Está completa (0 faltantes).

Contiene 52 valores distintos, cubriendo todas las semanas del año.

- La distribución es relativamente uniforme, con una media de 26.5 y una mediana de 27, lo que indica que las llegadas se concentran hacia la mitad del año.

- Las semanas con mayor cantidad de huéspedes se ubican entre la 27 y la 33, donde se observan los picos más altos (hasta 2.032 reservas).

- Las semanas iniciales y finales del año muestran menor actividad, especialmente la semana 1 y la semana 52, que presentan los valores más bajos del conjunto.

- La variable presenta baja asimetría (skew ≈ 0) y kurtosis negativa, indicando una distribución bastante plana y sin extremos marcados.

Como analisis preliminar se puede asumir que los picos en semanas centrales están asociados a vacaciones o temporadas turísticas específicas que incrementan la demanda.

---

### 8. arrival_date_day_of_month 

In [34]:
print(f"Conteo total: {df['arrival_date_day_of_month'].count()}")

print(f"\nCantidad de valores distintos: {df['arrival_date_day_of_month'].nunique()}")

print(f"\nNulos: {df['arrival_date_day_of_month'].isnull().sum()}")

print(f"\nEstadísticas:\n{df['arrival_date_day_of_month'].describe()}")

print(f"\nDatos y Cantidades:\n{df['arrival_date_day_of_month'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['arrival_date_day_of_month'].value_counts(normalize=True).round(4)}")





Conteo total: 25000

Cantidad de valores distintos: 31

Nulos: 0

Estadísticas:
count    25000.000000
mean        15.830240
std          8.806273
min          1.000000
25%          8.000000
50%         16.000000
75%         24.000000
max         31.000000
Name: arrival_date_day_of_month, dtype: float64

Datos y Cantidades:
arrival_date_day_of_month
1     784
2     881
3     792
4     802
5     927
6     740
7     747
8     749
9     879
10    767
11    741
12    845
13    788
14    785
15    877
16    857
17    945
18    877
19    825
20    806
21    808
22    762
23    752
24    860
25    885
26    838
27    810
28    796
29    747
30    859
31    469
Name: count, dtype: int64

Porcentajes:
arrival_date_day_of_month
17    0.0378
5     0.0371
25    0.0354
2     0.0352
9     0.0352
18    0.0351
15    0.0351
24    0.0344
30    0.0344
16    0.0343
12    0.0338
26    0.0335
19    0.0330
27    0.0324
21    0.0323
20    0.0322
4     0.0321
28    0.0318
3     0.0317
13    0.0315
14    0.0314


**Descripción**

La variable `arrival_date_day_of_month` representa el día del mes en el que el huésped llega al hotel, con valores entre 1 y 31.

**Faltantes**

Está completa (0 nulos).

Contiene 31 valores diferentes, cubriendo todos los días posibles del mes.

Los estadísticos muestran una distribución equilibrada, lo que nos marca que las llegadas de los huespedes son estables y no hay concentraciones en dias particulares. 

**Frecuencias por día**

- El dia con mayor cantidad de huéspedes es el día 17 con 945

- El día con menor cantidad de huéspedes es el día 31 con 469

- Los días más frecuentes rondan entre 3.5% y 3.7% del total.

- El día 31 tiene el porcentaje más bajo (1.88%).


Las llegadas se concentran en días centrales del mes, mientras que los extremos muestran menor actividad. Esto sugiere un patrón estable de reservas que favorece la mitad del mes.

---

### 9. arrival_date

In [48]:
print(f"Conteo total: {df['arrival_date'].count()}")

print(f"Cantidad de valores distintos: {df['arrival_date'].nunique()}")

print(f"Nulos: {df['arrival_date'].isnull().sum()}")


#valor mas frecuente y porcentaje

valor_mas_frecuente = df['arrival_date'].mode()[0]
frecuencia = df['arrival_date'].value_counts()[valor_mas_frecuente]
porcentaje = frecuencia / df['arrival_date'].count() * 100

print(f"\nValor más frecuente: {valor_mas_frecuente}")
print(f"Frecuencia: {frecuencia}")
print(f"Porcentaje del valor mas frecuente: {porcentaje:.2f}%")



Conteo total: 25000
Cantidad de valores distintos: 793
Nulos: 0

Valor más frecuente: 2023-12-05
Frecuencia: 88
Porcentaje del valor mas frecuente: 0.35%


**Descripción**

La variable arrival_date representa la fecha completa de llegada del huésped al hotel.

**Faltantes**

Está completa (0 nulos).

**Valores**

Contiene 793 valores diferentes, lo que muestra una alta variabilidad en las fechas registradas.

- Valor más frecuente: 2023‑12‑05

- Frecuencia: 88

- Porcentaje: 0.35%

Las fechas de llegada están muy dispersas y no se concentran en días específicos. El valor más frecuente representa solo el 0.35% del total, lo que muestra mucha variabilidad sin ningun pico.

---

### 10. stays_in_weekend_nights

In [52]:
print(f"Conteo total: {df['stays_in_weekend_nights'].count()}")
print(f"Cantidad de valores distintos: {df['stays_in_weekend_nights'].nunique()}")
print(f"Nulos: {df['stays_in_weekend_nights'].isnull().sum()}")

print(f"\nEstadísticas:\n{df['stays_in_weekend_nights'].describe()}")

print(f"\nDatos y Cantidades:\n{df['stays_in_weekend_nights'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['stays_in_weekend_nights'].value_counts(normalize=True).round(4)}")

print(f"\nMinimo y Maximo:\nMin: {df['stays_in_weekend_nights'].min()}  |  Max: {df['stays_in_weekend_nights'].max()}")


Conteo total: 25000
Cantidad de valores distintos: 12
Nulos: 0

Estadísticas:
count    25000.000000
mean         0.916640
std          0.991227
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max         14.000000
Name: stays_in_weekend_nights, dtype: float64

Datos y Cantidades:
stays_in_weekend_nights
0     10983
1      6486
2      6817
3       248
4       388
5        19
6        37
7         4
8        13
9         3
10        1
14        1
Name: count, dtype: int64

Porcentajes:
stays_in_weekend_nights
0     0.4393
2     0.2727
1     0.2594
4     0.0155
3     0.0099
6     0.0015
5     0.0008
8     0.0005
7     0.0002
9     0.0001
14    0.0000
10    0.0000
Name: proportion, dtype: float64

Minimo y Maximo:
Min: 0  |  Max: 14


**Descripción**

La variable `stays_in_weekend_nights` indica cuántas noches de fin de semana (viernes y sábado) permaneció el huésped en el hotel. Los valores observados van desde 0 hasta 14.

**Faltantes**
Está completa (0 nulos).

Contiene 12 valores diferentes, lo que muestra que la mayoría de los huéspedes tienen pocas noches de fin de semana registradas.

Los estadísticos muestran una distribución fuertemente concentrada en valores bajos, mostrando que los huéspedes no pasan noches de fin de semana en el hotel.

**Frecuencias por valor**

- El valor más frecuente es 0 noches, con 10.983 huéspedes.

- Los valores 1 y 2 noches también son comunes, con 6.486 y 6.817 respectivamente.

- A partir de 3 noches, las frecuencias caen.

**Porcentajes** 

- 0 noches representa el 43.93% del total.

- 2 noches: 27.27%

- 1 noche: 25.94%

- El resto de los valores tienen porcentajes menores al 2%, mostrando que estancias largas de fin de semana son muy poco frecuentes.

La mayoría de los huéspedes no se alojan durante fines de semana, y quienes lo hacen suelen quedarse solo 1 o 2 noches. Esto sugiere que el hotel recibe principalmente reservas de corta duración y con poca actividad en fines de semana.

---

### 11. stays_in_week_nights

In [53]:
print(f"Conteo total: {df['stays_in_week_nights'].count()}")
print(f"Cantidad de valores distintos: {df['stays_in_week_nights'].nunique()}")
print(f"Nulos: {df['stays_in_week_nights'].isnull().sum()}")

print(f"\nEstadísticas:\n{df['stays_in_week_nights'].describe()}")

print(f"\nDatos y Cantidades:\n{df['stays_in_week_nights'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['stays_in_week_nights'].value_counts(normalize=True).round(4)}")

print(f"\nMinimo y Maximo:\nMin: {df['stays_in_week_nights'].min()}  |  Max: {df['stays_in_week_nights'].max()}")


Conteo total: 25000
Cantidad de valores distintos: 25
Nulos: 0

Estadísticas:
count    25000.000000
mean         2.479680
std          1.878629
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         34.000000
Name: stays_in_week_nights, dtype: float64

Datos y Cantidades:
stays_in_week_nights
0     1584
1     6368
2     7246
3     4587
4     1978
5     2262
6      313
7      192
8      121
9       44
10     214
11      17
12       9
13       3
14       5
15      22
16       7
17       1
18       1
19      11
20       4
21       5
22       3
24       2
34       1
Name: count, dtype: int64

Porcentajes:
stays_in_week_nights
2     0.2898
1     0.2547
3     0.1835
5     0.0905
4     0.0791
0     0.0634
6     0.0125
10    0.0086
7     0.0077
8     0.0048
9     0.0018
15    0.0009
11    0.0007
19    0.0004
12    0.0004
16    0.0003
21    0.0002
14    0.0002
20    0.0002
13    0.0001
22    0.0001
24    0.0001
18    0.0000
34    0.0000
17    0.0000


**Descripción**

La variable `stays_in_week_nights` representa la cantidad de noches de días de semana (lunes a jueves) que el  huésped permaneció en el hotel. Los valores observados van desde 0 hasta 34.

**Faltantes**

Está completa (0 nulos).

Contiene 25 valores diferentes, lo que muestra una mayor variabilidad respecto a las noches de fin de semana.

Los estadísticos muestran una distribución concentrada en valores bajos, mostrando estancias cortas durante días de semana.

**Frecuencias por valor**

- Los valores más frecuentes son 2 noches (28.98%), 1 noche (25.47%) y 3 noches (18.35%).

- A partir de 4 noches, las frecuencias disminuyen

- Los valores altos (17, 18, 24, 34) aparecen en cantidades mínimas, entre 1 y 2 registros.

**Porcentajes**

- Más del 70% de los huéspedes se alojan entre 1 y 3 noches.

- Las estadias de 4 a 6 noches representan porcentajes moderados.

- Las estadias largas (más de 10 noches) son extremadamente raras, con porcentajes cercanos a 0%.

La distribución muestra que los huéspedes suelen tener estadias cortas durante días de semana, con muy pocos casos de estadías prolongadas. 

---